# Phase 1: GDC Data Quality Check

Notebook này lọc các sample GDC đã qua bước cleaning, giữ lại sample không bị outlier về library size và số gene phát hiện được, sau đó ghi expression protein-coding đã thêm `log2_tpm` vào layer analysis trên HDFS.

Nguyên tắc chạy:
- Chỉ đọc dữ liệu refined, không sửa raw/refined.
- Ưu tiên đọc Hive table `gdc.quality_check` và `gdc.gdc_counts_clean_protein_coding` nếu đã đăng ký.
- Nếu Hive table chưa có, tự fallback sang Parquet refined trên HDFS.
- Chỉ ghi output vào `hdfs://master11:9000/drugtarget/data/analysis/gdc_qc_pass_expression`.

In [1]:
from pyspark.sql import SparkSession, functions as F

# Khai báo các đường dẫn HDFS và tên bảng phục vụ Phase 1.
HDFS_BASE = "hdfs://master11:9000"
QC_TABLE = "gdc.quality_check"
EXPR_TABLE = "gdc.gdc_counts_clean_protein_coding"
QC_PATH = f"{HDFS_BASE}/drugtarget/data/refined/gdc/quality_check"
EXPR_PATH = f"{HDFS_BASE}/drugtarget/data/refined/gdc/gdc_counts_clean_protein_coding"
OUTPUT_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/gdc_qc_pass_expression"

# Tạo SparkSession để đọc dữ liệu refined và ghi kết quả analysis.
spark = (
    SparkSession.builder.appName("drugtarget-gdc-phase1-quality-check")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .enableHiveSupport()
    .getOrCreate()
)

# Giảm log Spark để output notebook dễ đọc hơn.
spark.sparkContext.setLogLevel("WARN")

print(f"Spark master: {spark.sparkContext.master}")
print(f"Output Phase 1: {OUTPUT_PATH}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/03 07:26:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark master: local[*]
Output Phase 1: hdfs://master11:9000/drugtarget/data/analysis/gdc_qc_pass_expression


In [2]:
def read_table_or_parquet(table_name: str, parquet_path: str):
    """Đọc Hive table nếu có; nếu chưa có thì đọc Parquet refined trên HDFS."""
    try:
        frame = spark.table(table_name)
        print(f"Đọc Hive table thành công: {table_name}")
        return frame
    except Exception as exc:
        print(f"Không đọc được Hive table {table_name}; chuyển sang Parquet refined.")
        print(f"Lý do: {type(exc).__name__}: {exc}")
        print(f"Đọc Parquet: {parquet_path}")
        return spark.read.parquet(parquet_path)


def require_columns(frame, required_columns, frame_name: str) -> None:
    """Dừng pipeline nếu DataFrame thiếu cột bắt buộc."""
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    if missing_columns:
        missing_text = ", ".join(missing_columns)
        raise ValueError(f"{frame_name} thiếu cột bắt buộc: {missing_text}")


# Đọc hai input chính của Phase 1.
qc = read_table_or_parquet(QC_TABLE, QC_PATH)
expr = read_table_or_parquet(EXPR_TABLE, EXPR_PATH)

# Kiểm tra schema tối thiểu trước khi xử lý.
QC_REQUIRED_COLUMNS = [
    "sample_id",
    "case_id",
    "sample_group",
    "is_outlier_library_size",
    "is_outlier_detected_genes",
]
EXPR_REQUIRED_COLUMNS = [
    "sample_id",
    "case_id",
    "sample_group",
    "gene_id_base",
    "gene_name",
    "tpm",
]

require_columns(qc, QC_REQUIRED_COLUMNS, "quality_check")
require_columns(expr, EXPR_REQUIRED_COLUMNS, "gdc_counts_clean_protein_coding")

print("Schema quality_check:")
qc.printSchema()
print("Schema gdc_counts_clean_protein_coding:")
expr.printSchema()

26/06/03 07:26:58 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/03 07:26:58 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


26/06/03 07:27:01 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/06/03 07:27:01 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dis@100.82.104.4


26/06/03 07:27:02 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


26/06/03 07:27:02 WARN ObjectStore: Failed to get database gdc, returning NoSuchObjectException


Không đọc được Hive table gdc.quality_check; chuyển sang Parquet refined.
Lý do: AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `gdc`.`quality_check` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [gdc, quality_check], [], false

Đọc Parquet: hdfs://master11:9000/drugtarget/data/refined/gdc/quality_check


26/06/03 07:27:06 WARN ObjectStore: Failed to get database gdc, returning NoSuchObjectException


Không đọc được Hive table gdc.gdc_counts_clean_protein_coding; chuyển sang Parquet refined.
Lý do: AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `gdc`.`gdc_counts_clean_protein_coding` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [gdc, gdc_counts_clean_protein_coding], [], false

Đọc Parquet: hdfs://master11:9000/drugtarget/data/refined/gdc/gdc_counts_clean_protein_coding


Schema quality_check:
root
 |-- file_id: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- sample_group: string (nullable = true)
 |-- total_raw_count: long (nullable = true)
 |-- num_detected_genes_raw_gt_0: long (nullable = true)
 |-- num_detected_genes_tpm_gt_1: long (nullable = true)
 |-- median_tpm: double (nullable = true)
 |-- pct_zero_genes: double (nullable = true)
 |-- num_protein_coding_genes: long (nullable = true)
 |-- is_outlier_library_size: boolean (nullable = true)
 |-- is_outlier_detected_genes: boolean (nullable = true)

Schema gdc_counts_clean_protein_coding:
root
 |-- file_id: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_type: stri

In [3]:
# Gom QC về mức sample để loại sample nếu bất kỳ dòng QC nào bị outlier.
qc_sample_status = (
    qc.select(
        "sample_id",
        "case_id",
        "sample_group",
        (
            (F.col("is_outlier_library_size") == F.lit(True))
            | (F.col("is_outlier_detected_genes") == F.lit(True))
        )
        .cast("int")
        .alias("has_outlier_row"),
    )
    .groupBy("sample_id", "case_id", "sample_group")
    .agg(
        F.max("has_outlier_row").alias("has_any_outlier"),
        F.count("*").alias("qc_row_count"),
    )
)

# Giữ lại sample không có bất kỳ outlier nào về tổng count hoặc số gene phát hiện được.
qc_pass = qc_sample_status.filter(F.col("has_any_outlier") == F.lit(0)).select(
    "sample_id", "case_id", "sample_group"
)

# In số sample pass QC theo nhóm để theo dõi chất lượng input.
qc_pass_counts = qc_pass.groupBy("sample_group").count().orderBy("sample_group")
print("Số sample pass QC theo sample_group:")
qc_pass_counts.show(truncate=False)

# So sánh nhẹ với số lượng dự kiến hiện tại; không dừng nếu refined được cập nhật sau này.
expected_pass_counts = {"normal": 59, "tumor": 520}
actual_pass_counts = {row["sample_group"]: row["count"] for row in qc_pass_counts.collect()}
if actual_pass_counts == expected_pass_counts:
    print("Số sample pass QC khớp với dự kiến hiện tại: normal=59, tumor=520")
else:
    print(f"Cảnh báo: số sample pass QC khác dự kiến hiện tại: {actual_pass_counts}")

Số sample pass QC theo sample_group:


+------------+-----+
|sample_group|count|
+------------+-----+
|normal      |59   |
|tumor       |520  |
+------------+-----+



Số sample pass QC khớp với dự kiến hiện tại: normal=59, tumor=520


In [4]:
# Join expression với danh sách sample pass QC và chỉ giữ schema analysis cần dùng.
OUTPUT_COLUMNS = [
    "sample_id",
    "case_id",
    "sample_group",
    "gene_id_base",
    "gene_name",
    "tpm",
    "log2_tpm",
]

gdc_qc_pass_expression = (
    expr.join(qc_pass, on=["sample_id", "case_id", "sample_group"], how="inner")
    .select(
        "sample_id",
        "case_id",
        "sample_group",
        "gene_id_base",
        "gene_name",
        F.col("tpm").cast("double").alias("tpm"),
    )
    .withColumn("log2_tpm", F.log2(F.col("tpm") + F.lit(1.0)))
    .select(*OUTPUT_COLUMNS)
    .cache()
)

# Tính count trước khi ghi để phát hiện sớm lỗi join hoặc dữ liệu rỗng.
output_row_count = gdc_qc_pass_expression.count()
if output_row_count == 0:
    raise ValueError("Output Phase 1 rỗng; cần kiểm tra input QC/expression.")

print("Schema output Phase 1:")
gdc_qc_pass_expression.printSchema()
print(f"Số dòng output Phase 1: {output_row_count}")

26/06/03 07:27:41 WARN MemoryStore: Not enough space to cache rdd_42_3 in memory! (computed 75.6 MiB so far)
26/06/03 07:27:41 WARN BlockManager: Persisting block rdd_42_3 to disk instead.


26/06/03 07:27:44 WARN MemoryStore: Not enough space to cache rdd_42_3 in memory! (computed 75.6 MiB so far)


26/06/03 07:27:49 WARN MemoryStore: Not enough space to cache rdd_42_3 in memory! (computed 27.7 MiB so far)


Schema output Phase 1:
root
 |-- sample_id: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_group: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- tpm: double (nullable = true)
 |-- log2_tpm: double (nullable = true)

Số dòng output Phase 1: 11747016


In [5]:
# Ghi output Phase 1 vào layer analysis, không ghi vào raw/refined.
gdc_qc_pass_expression.write.mode("overwrite").option("compression", "snappy").parquet(OUTPUT_PATH)
print(f"Đã ghi output Phase 1: {OUTPUT_PATH}")

26/06/03 07:27:57 WARN MemoryStore: Not enough space to cache rdd_42_3 in memory! (computed 27.7 MiB so far)


Đã ghi output Phase 1: hdfs://master11:9000/drugtarget/data/analysis/gdc_qc_pass_expression


In [6]:
# Kiểm tra output path tồn tại trên HDFS sau khi ghi.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
output_hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(OUTPUT_PATH)
output_fs = output_hdfs_path.getFileSystem(hadoop_conf)
if not output_fs.exists(output_hdfs_path):
    raise AssertionError(f"Output path chưa tồn tại trên HDFS: {OUTPUT_PATH}")

# Đọc lại output để xác nhận schema, log2_tpm và điều kiện không còn outlier.
output_df = spark.read.parquet(OUTPUT_PATH)
if output_df.columns != OUTPUT_COLUMNS:
    raise AssertionError(f"Schema output không đúng: {output_df.columns}")

bad_log2_count = output_df.filter(F.col("tpm").isNotNull() & F.col("log2_tpm").isNull()).count()
if bad_log2_count != 0:
    raise AssertionError(f"Có {bad_log2_count} dòng tpm không null nhưng log2_tpm null.")

bad_sample_keys = (
    qc.filter(
        (F.col("is_outlier_library_size") == F.lit(True))
        | (F.col("is_outlier_detected_genes") == F.lit(True))
    )
    .select("sample_id", "case_id", "sample_group")
    .distinct()
)
bad_output_sample_count = (
    output_df.select("sample_id", "case_id", "sample_group")
    .distinct()
    .join(bad_sample_keys, on=["sample_id", "case_id", "sample_group"], how="inner")
    .count()
)
if bad_output_sample_count != 0:
    raise AssertionError(f"Output còn {bad_output_sample_count} sample outlier.")

print("Kiểm tra output Phase 1 hoàn tất.")
print("Số sample trong output theo sample_group:")
output_df.select("sample_id", "case_id", "sample_group").distinct().groupBy("sample_group").count().orderBy("sample_group").show(truncate=False)

Kiểm tra output Phase 1 hoàn tất.
Số sample trong output theo sample_group:


+------------+-----+
|sample_group|count|
+------------+-----+
|normal      |59   |
|tumor       |520  |
+------------+-----+



In [7]:
# Giải phóng cache sau khi hoàn tất Phase 1.
gdc_qc_pass_expression.unpersist()
print("Hoàn tất Phase 1: GDC Data Quality Check")

Hoàn tất Phase 1: GDC Data Quality Check
